<a href="https://colab.research.google.com/github/daniya-sohail26/Langchain_Course_Projects/blob/main/healthcaretriagesystem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install LangChain, LangGraph, OpenAI SDK (for OpenRouter), and NLP dependencies
!pip install langchain langchain-openai langgraph scispacy requests
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

  Using cached https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz (14.8 MB)
  Preparing metadata (setup.py) ... done


In [2]:
!pip install -U langchain-openrouter

In [3]:
import getpass
import os

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")

Enter your OpenRouter API key: ··········


In [4]:
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(
    model="openrouter/free",
    temperature=0
)

In [10]:
from typing import TypedDict, Annotated, List
from langgraph.graph.message import AnyMessage, add_messages


class CustomState(TypedDict):
    messages: Annotated[List[AnyMessage], add_messages]

In [11]:
import spacy
import requests
from langchain.tools import tool
#USING SCISPACY NLP
nlp = spacy.load("en_core_sci_sm")

@tool
def extract_symptoms(text: str)->list:
    """Always use this tool FIRST. Use it to extract medical symptoms from the patient's raw complaint."""
    print(f"\n[Tool Execution] Extracting symptoms from: '{text}'")
    doc = nlp(text)
    symptoms = list(set([ent.text.lower() for ent in doc.ents]))
    return symptoms

@tool
def check_adverse_events(medication: str, symptom: str)-> str:
    """Use this tool to check if a specific symptom is a known side effect of a patient's medication."""
    print(f"\n[Tool Execution] Checking FDA for {symptom} caused by {medication}...")
    url = f'https://api.fda.gov/drug/event.json?search=patient.drug.medicinalproduct:"{medication}"+AND+patient.reaction.reactionmeddrapt:"{symptom}"&limit=1'
    try:
        res = requests.get(url)
        if res.status_code == 200:
            return f"WARNING: '{symptom}' is a reported adverse event for '{medication}'."
    except Exception:
        pass
    return f"Safe: No major FDA reports linking '{symptom}' to '{medication}'."

@tool
def calculate_triage_level(age: int, sex: str, symptoms: str) -> str:
    """Use this tool to get a medical diagnosis and triage severity. Input symptoms as a comma-separated string."""
    print("\n[Tool Execution] Calling EndlessMedical for triage calculation...")
    base_url = "https://api.endlessmedical.com/v1/dx"

    try:
        # Init Session
        session_id = requests.get(f"{base_url}/InitSession").json().get("SessionID")
        requests.post(f"{base_url}/AcceptTermsOfUse?SessionID={session_id}&passphrase=I have read, understood and I accept and agree to comply with the Terms of Use of EndlessMedicalAPI and Endless Medical services. The Terms of Use are available on endlessmedical.com")

        # Add Patient State
        requests.post(f"{base_url}/UpdateFeature?SessionID={session_id}&name=Age&value={age}")
        requests.post(f"{base_url}/UpdateFeature?SessionID={session_id}&name=Gender&value={sex.capitalize()}")

        # Add Symptoms
        for symp in symptoms.split(","):
            requests.post(f"{base_url}/UpdateFeature?SessionID={session_id}&name={symp.strip()}&value=1")

        # Analyze
        analyze_res = requests.get(f"{base_url}/Analyze?SessionID={session_id}").json()
        diseases = analyze_res.get("Diseases", {})
        top_3 = dict(list(diseases.items())[:3])

        return f"Top Diagnoses: {list(top_3.keys())}. Action: Triage Consultation Advised."
    except Exception as e:
        return "Triage calculation failed due to API error."


tools = [extract_symptoms, check_adverse_events, calculate_triage_level]

In [12]:
from langchain.agents import create_agent
from langchain.messages import SystemMessage
from langgraph.checkpoint.memory import InMemorySaver

system_prompt = SystemMessage(content="""
You are the Lead Triage Orchestrator at Colab Urgent Care.
You are given a patient's profile (age, sex, current medications, and their complaint).

You MUST follow this exact sequence using your tools:
1. Extract the symptoms from the complaint.
2. If the patient is on medications, check IF the extracted symptoms are adverse events of those medications.
3. Calculate the triage level and differential diagnosis using the extracted symptoms.
4. Finally, write a brief, professional Clinical Summary combining all your findings.
""")

agent_executor = create_agent(model, tools, checkpointer=InMemorySaver(), state_schema=CustomState)

In [14]:
patient_prompt = """
Patient Age: 45
Patient Sex: female
Current Medications: lisinopril
Patient Complaint: "My head is pounding, I feel nauseous, and my vision is a bit blurry."
"""

print("Starting Autonomous Triage...\n")

# Stream the agent's thought process and actions
for event in agent_executor.stream(
    {"messages": [("user", patient_prompt)]},
    config={"configurable": {"thread_id": "1"}},
    stream_mode="values"
):
    # Print the last message generated in the state
    message = event["messages"][-1]

    # If it's an AI message, it might be a tool call or the final answer
    if message.type == "ai":
        if message.tool_calls:
            for tc in message.tool_calls:
                print(f"Main Agent is delegating to -> {tc['name']}")
        elif message.content:
            print("\n================ FINAL CLINICAL NOTE ================")
            print(message.content)
            print("=====================================================")

Starting Autonomous Triage...

Main Agent is delegating to -> extract_symptoms

[Tool Execution] Extracting symptoms from: 'My head is pounding, I feel nauseous, and my vision is a bit blurry.'
Main Agent is delegating to -> check_adverse_events
Main Agent is delegating to -> check_adverse_events
Main Agent is delegating to -> check_adverse_events

[Tool Execution] Checking FDA for pounding head caused by lisinopril...

[Tool Execution] Checking FDA for nausea caused by lisinopril...

[Tool Execution] Checking FDA for blurry vision caused by lisinopril...
Main Agent is delegating to -> calculate_triage_level

[Tool Execution] Calling EndlessMedical for triage calculation...
Main Agent is delegating to -> calculate_triage_level

[Tool Execution] Calling EndlessMedical for triage calculation...

================ FINAL CLINICAL NOTE ================
**Summary of the case**

| Item | Details |
|------|---------|
| Age | 45 years |
| Sex | Female |
| Current medication | Lisinopril (an ACE‑